# 30 LLM API Calls — Hands-on Python Notebook

**Goal:** Learn to invoke 30 model/API configurations across 11 LLM providers, step by step.

**Important:** Model IDs, access tiers, endpoints, pricing, and availability change. The IDs below are illustrative defaults; override any model with an environment variable (listed in each example) if your account offers a different model. Some providers expose the same underlying model through their own API. **These are 30 API examples, not 30 independently developed foundation models.**

Run the setup cells, add API keys for the providers you want to use, then run individual examples. **Do not run all 30 unless you intend to incur API charges.**

## 1. Install dependencies

Works in Google Colab, JupyterLab, and VS Code notebooks. Restart the kernel if your environment requests it.

In [ ]:
%pip -q install -U openai anthropic google-genai python-dotenv pandas

## 2. Configure API keys safely

Use environment variables or a local `.env` file. **Never commit API keys to GitHub.** In Colab, you may also set environment variables using the Secrets interface. The examples skip calls when the required key is missing.

In [ ]:
import os, json, time
from dotenv import load_dotenv
from openai import OpenAI
import pandas as pd
load_dotenv()

# Local .env example (create it yourself; do not commit):
# OPENAI_API_KEY=...
# ANTHROPIC_API_KEY=...
# GEMINI_API_KEY=...
# GROQ_API_KEY=...
# TOGETHER_API_KEY=...
# FIREWORKS_API_KEY=...
# OPENROUTER_API_KEY=...
# MISTRAL_API_KEY=...
# DEEPSEEK_API_KEY=...
# CEREBRAS_API_KEY=...
# XAI_API_KEY=...

PROMPT = "Explain retrieval augmented generation in two simple sentences."
TEMPERATURE = 0.2
MAX_TOKENS = 250
RESULTS = []
print("Setup complete. Configure only the provider keys you plan to use.")

## 3. Reusable functions

Most providers support an OpenAI-compatible chat-completions endpoint. Anthropic and Gemini examples use their native SDKs. These functions show the same pattern: **authenticate → send messages → read response → log latency**.

In [ ]:
PROVIDERS = {
    "OpenAI": ("OPENAI_API_KEY", None),
    "Groq": ("GROQ_API_KEY", "https://api.groq.com/openai/v1"),
    "Together AI": ("TOGETHER_API_KEY", "https://api.together.xyz/v1"),
    "Fireworks AI": ("FIREWORKS_API_KEY", "https://api.fireworks.ai/inference/v1"),
    "OpenRouter": ("OPENROUTER_API_KEY", "https://openrouter.ai/api/v1"),
    "Mistral": ("MISTRAL_API_KEY", "https://api.mistral.ai/v1"),
    "DeepSeek": ("DEEPSEEK_API_KEY", "https://api.deepseek.com"),
    "Cerebras": ("CEREBRAS_API_KEY", "https://api.cerebras.ai/v1"),
    "xAI": ("XAI_API_KEY", "https://api.x.ai/v1"),
}

def record(provider, model, response, elapsed):
    RESULTS.append({"provider": provider, "model": model, "seconds": round(elapsed, 2), "response": response})
    print(f"Provider: {provider} | Model: {model} | Latency: {elapsed:.2f}s")
    print(response)
    return response

def call_compatible(provider, model, prompt=PROMPT, temperature=TEMPERATURE):
    key_name, base_url = PROVIDERS[provider]
    key = os.getenv(key_name)
    if not key:
        print(f"SKIPPED: Set {key_name} to run this example.")
        return None
    client = OpenAI(api_key=key, **({"base_url": base_url} if base_url else {}), timeout=45, max_retries=1)
    started = time.perf_counter()
    try:
        result = client.chat.completions.create(
            model=model,
            messages=[{"role": "system", "content": "You are a concise technical instructor."},
                      {"role": "user", "content": prompt}],
            temperature=temperature,
            max_tokens=MAX_TOKENS,
        )
        answer = result.choices[0].message.content or "[No text returned]"
        return record(provider, model, answer, time.perf_counter()-started)
    except Exception as exc:
        print(f"REQUEST FAILED ({provider}/{model}): {type(exc).__name__}: {exc}")
        print("Check model access, model ID, billing, rate limits, and provider API compatibility.")
        return None

def call_anthropic(model, prompt=PROMPT):
    key = os.getenv("ANTHROPIC_API_KEY")
    if not key:
        print("SKIPPED: Set ANTHROPIC_API_KEY.")
        return None
    import anthropic
    started = time.perf_counter()
    try:
        client = anthropic.Anthropic(api_key=key, timeout=45.0, max_retries=1)
        response = client.messages.create(model=model, max_tokens=MAX_TOKENS,
            system="You are a concise technical instructor.",
            messages=[{"role": "user", "content": prompt}])
        answer = "\n".join(block.text for block in response.content if block.type == "text")
        return record("Anthropic", model, answer, time.perf_counter()-started)
    except Exception as exc:
        print(f"REQUEST FAILED (Anthropic/{model}): {type(exc).__name__}: {exc}")
        return None

def call_gemini(model, prompt=PROMPT):
    key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if not key:
        print("SKIPPED: Set GEMINI_API_KEY or GOOGLE_API_KEY.")
        return None
    from google import genai
    from google.genai import types
    started = time.perf_counter()
    try:
        client = genai.Client(api_key=key)
        response = client.models.generate_content(model=model, contents=prompt,
            config=types.GenerateContentConfig(temperature=TEMPERATURE, max_output_tokens=MAX_TOKENS))
        answer = response.text or "[No text returned]"
        return record("Google Gemini", model, answer, time.perf_counter()-started)
    except Exception as exc:
        print(f"REQUEST FAILED (Gemini/{model}): {type(exc).__name__}: {exc}")
        return None


## 4. Run 30 individual LLM API examples

Each cell is independent. Change `PROMPT` globally or pass a different prompt to an individual call. For each example, you can override the model ID with `MODEL_01`, `MODEL_02`, … `MODEL_30` environment variables. **Some listed IDs may be retired or unavailable on your account; consult the provider model catalog if a request fails.**

### Example 01: OpenAI GPT-4.1

**Provider:** OpenAI  
**Illustrative model ID:** `gpt-4.1`  
**Override:** `MODEL_01`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 01: OpenAI GPT-4.1
response_01 = call_compatible("OpenAI", os.getenv("MODEL_01", "gpt-4.1"))

### Example 02: OpenAI GPT-4.1 mini

**Provider:** OpenAI  
**Illustrative model ID:** `gpt-4.1-mini`  
**Override:** `MODEL_02`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 02: OpenAI GPT-4.1 mini
response_02 = call_compatible("OpenAI", os.getenv("MODEL_02", "gpt-4.1-mini"))

### Example 03: OpenAI GPT-4.1 nano

**Provider:** OpenAI  
**Illustrative model ID:** `gpt-4.1-nano`  
**Override:** `MODEL_03`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 03: OpenAI GPT-4.1 nano
response_03 = call_compatible("OpenAI", os.getenv("MODEL_03", "gpt-4.1-nano"))

### Example 04: OpenAI GPT-4o mini

**Provider:** OpenAI  
**Illustrative model ID:** `gpt-4o-mini`  
**Override:** `MODEL_04`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 04: OpenAI GPT-4o mini
response_04 = call_compatible("OpenAI", os.getenv("MODEL_04", "gpt-4o-mini"))

### Example 05: Anthropic Claude Sonnet 4.5

**Provider:** Anthropic  
**Illustrative model ID:** `claude-sonnet-4-5`  
**Override:** `MODEL_05`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 05: Anthropic Claude Sonnet 4.5
response_05 = call_anthropic(os.getenv("MODEL_05", "claude-sonnet-4-5"))

### Example 06: Anthropic Claude Haiku 4.5

**Provider:** Anthropic  
**Illustrative model ID:** `claude-haiku-4-5`  
**Override:** `MODEL_06`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 06: Anthropic Claude Haiku 4.5
response_06 = call_anthropic(os.getenv("MODEL_06", "claude-haiku-4-5"))

### Example 07: Anthropic Claude Opus 4.1

**Provider:** Anthropic  
**Illustrative model ID:** `claude-opus-4-1`  
**Override:** `MODEL_07`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 07: Anthropic Claude Opus 4.1
response_07 = call_anthropic(os.getenv("MODEL_07", "claude-opus-4-1"))

### Example 08: Google Gemini 2.5 Pro

**Provider:** Google Gemini  
**Illustrative model ID:** `gemini-2.5-pro`  
**Override:** `MODEL_08`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 08: Google Gemini 2.5 Pro
response_08 = call_gemini(os.getenv("MODEL_08", "gemini-2.5-pro"))

### Example 09: Google Gemini 2.5 Flash

**Provider:** Google Gemini  
**Illustrative model ID:** `gemini-2.5-flash`  
**Override:** `MODEL_09`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 09: Google Gemini 2.5 Flash
response_09 = call_gemini(os.getenv("MODEL_09", "gemini-2.5-flash"))

### Example 10: Google Gemini 2.5 Flash Lite

**Provider:** Google Gemini  
**Illustrative model ID:** `gemini-2.5-flash-lite`  
**Override:** `MODEL_10`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 10: Google Gemini 2.5 Flash Lite
response_10 = call_gemini(os.getenv("MODEL_10", "gemini-2.5-flash-lite"))

### Example 11: Groq-hosted Llama 3.3 70B

**Provider:** Groq  
**Illustrative model ID:** `llama-3.3-70b-versatile`  
**Override:** `MODEL_11`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 11: Groq-hosted Llama 3.3 70B
response_11 = call_compatible("Groq", os.getenv("MODEL_11", "llama-3.3-70b-versatile"))

### Example 12: Groq-hosted GPT-OSS 120B

**Provider:** Groq  
**Illustrative model ID:** `openai/gpt-oss-120b`  
**Override:** `MODEL_12`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 12: Groq-hosted GPT-OSS 120B
response_12 = call_compatible("Groq", os.getenv("MODEL_12", "openai/gpt-oss-120b"))

### Example 13: Groq-hosted GPT-OSS 20B

**Provider:** Groq  
**Illustrative model ID:** `openai/gpt-oss-20b`  
**Override:** `MODEL_13`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 13: Groq-hosted GPT-OSS 20B
response_13 = call_compatible("Groq", os.getenv("MODEL_13", "openai/gpt-oss-20b"))

### Example 14: Together-hosted Llama 3.3 70B

**Provider:** Together AI  
**Illustrative model ID:** `meta-llama/Llama-3.3-70B-Instruct-Turbo`  
**Override:** `MODEL_14`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 14: Together-hosted Llama 3.3 70B
response_14 = call_compatible("Together AI", os.getenv("MODEL_14", "meta-llama/Llama-3.3-70B-Instruct-Turbo"))

### Example 15: Together-hosted Qwen 2.5 72B

**Provider:** Together AI  
**Illustrative model ID:** `Qwen/Qwen2.5-72B-Instruct-Turbo`  
**Override:** `MODEL_15`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 15: Together-hosted Qwen 2.5 72B
response_15 = call_compatible("Together AI", os.getenv("MODEL_15", "Qwen/Qwen2.5-72B-Instruct-Turbo"))

### Example 16: Together-hosted DeepSeek R1

**Provider:** Together AI  
**Illustrative model ID:** `deepseek-ai/DeepSeek-R1`  
**Override:** `MODEL_16`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 16: Together-hosted DeepSeek R1
response_16 = call_compatible("Together AI", os.getenv("MODEL_16", "deepseek-ai/DeepSeek-R1"))

### Example 17: Fireworks-hosted Llama 3.3 70B

**Provider:** Fireworks AI  
**Illustrative model ID:** `accounts/fireworks/models/llama-v3p3-70b-instruct`  
**Override:** `MODEL_17`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 17: Fireworks-hosted Llama 3.3 70B
response_17 = call_compatible("Fireworks AI", os.getenv("MODEL_17", "accounts/fireworks/models/llama-v3p3-70b-instruct"))

### Example 18: Fireworks-hosted DeepSeek R1

**Provider:** Fireworks AI  
**Illustrative model ID:** `accounts/fireworks/models/deepseek-r1`  
**Override:** `MODEL_18`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 18: Fireworks-hosted DeepSeek R1
response_18 = call_compatible("Fireworks AI", os.getenv("MODEL_18", "accounts/fireworks/models/deepseek-r1"))

### Example 19: Fireworks-hosted Qwen3

**Provider:** Fireworks AI  
**Illustrative model ID:** `accounts/fireworks/models/qwen3-235b-a22b`  
**Override:** `MODEL_19`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 19: Fireworks-hosted Qwen3
response_19 = call_compatible("Fireworks AI", os.getenv("MODEL_19", "accounts/fireworks/models/qwen3-235b-a22b"))

### Example 20: OpenRouter-routed GPT-4.1 mini

**Provider:** OpenRouter  
**Illustrative model ID:** `openai/gpt-4.1-mini`  
**Override:** `MODEL_20`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 20: OpenRouter-routed GPT-4.1 mini
response_20 = call_compatible("OpenRouter", os.getenv("MODEL_20", "openai/gpt-4.1-mini"))

### Example 21: OpenRouter-routed Claude Sonnet 4.5

**Provider:** OpenRouter  
**Illustrative model ID:** `anthropic/claude-sonnet-4.5`  
**Override:** `MODEL_21`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 21: OpenRouter-routed Claude Sonnet 4.5
response_21 = call_compatible("OpenRouter", os.getenv("MODEL_21", "anthropic/claude-sonnet-4.5"))

### Example 22: OpenRouter-routed Gemini 2.5 Flash

**Provider:** OpenRouter  
**Illustrative model ID:** `google/gemini-2.5-flash`  
**Override:** `MODEL_22`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 22: OpenRouter-routed Gemini 2.5 Flash
response_22 = call_compatible("OpenRouter", os.getenv("MODEL_22", "google/gemini-2.5-flash"))

### Example 23: Mistral Small

**Provider:** Mistral  
**Illustrative model ID:** `mistral-small-latest`  
**Override:** `MODEL_23`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 23: Mistral Small
response_23 = call_compatible("Mistral", os.getenv("MODEL_23", "mistral-small-latest"))

### Example 24: Mistral Medium

**Provider:** Mistral  
**Illustrative model ID:** `mistral-medium-latest`  
**Override:** `MODEL_24`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 24: Mistral Medium
response_24 = call_compatible("Mistral", os.getenv("MODEL_24", "mistral-medium-latest"))

### Example 25: Mistral Large

**Provider:** Mistral  
**Illustrative model ID:** `mistral-large-latest`  
**Override:** `MODEL_25`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 25: Mistral Large
response_25 = call_compatible("Mistral", os.getenv("MODEL_25", "mistral-large-latest"))

### Example 26: DeepSeek Chat

**Provider:** DeepSeek  
**Illustrative model ID:** `deepseek-chat`  
**Override:** `MODEL_26`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 26: DeepSeek Chat
response_26 = call_compatible("DeepSeek", os.getenv("MODEL_26", "deepseek-chat"))

### Example 27: DeepSeek Reasoner

**Provider:** DeepSeek  
**Illustrative model ID:** `deepseek-reasoner`  
**Override:** `MODEL_27`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 27: DeepSeek Reasoner
response_27 = call_compatible("DeepSeek", os.getenv("MODEL_27", "deepseek-reasoner"))

### Example 28: Cerebras-hosted Llama 3.3 70B

**Provider:** Cerebras  
**Illustrative model ID:** `llama-3.3-70b`  
**Override:** `MODEL_28`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 28: Cerebras-hosted Llama 3.3 70B
response_28 = call_compatible("Cerebras", os.getenv("MODEL_28", "llama-3.3-70b"))

### Example 29: Cerebras-hosted Qwen3 32B

**Provider:** Cerebras  
**Illustrative model ID:** `qwen-3-32b`  
**Override:** `MODEL_29`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 29: Cerebras-hosted Qwen3 32B
response_29 = call_compatible("Cerebras", os.getenv("MODEL_29", "qwen-3-32b"))

### Example 30: xAI Grok 3 Mini

**Provider:** xAI  
**Illustrative model ID:** `grok-3-mini`  
**Override:** `MODEL_30`  

This example sends the shared RAG question, prints the generated text, and logs response latency. The model must be enabled for your account.

In [ ]:
# Example 30: xAI Grok 3 Mini
response_30 = call_compatible("xAI", os.getenv("MODEL_30", "grok-3-mini"))

## 5. Compare successful responses

Run after testing any number of providers. Missing-key and failed calls are not added to the table.

In [ ]:
if RESULTS:
    display(pd.DataFrame(RESULTS)[["provider", "model", "seconds", "response"]])
else:
    print("No successful API calls yet. Add a provider key and run an example.")

## 6. Practical assignments

1. Change `PROMPT` to: *Explain HNSW in five steps.* Compare three providers.
2. Compare response time and quality for the same prompt; repeat multiple times to account for latency variation.
3. Create a JSON-format prompt and validate its output using `json.loads`.
4. Test a Python-code-generation prompt, then execute the generated code only after reviewing it.
5. Build a small benchmark dataset of 20 prompts with reference answers. Score relevance, factuality, latency, and cost.

### API safety and troubleshooting
- Never publish keys or store them in notebooks shared with students.
- A missing API key is handled by skipping the example.
- 401/403: authentication or access; 404: model ID or endpoint; 429: rate limits or quota; 400: unsupported parameters or invalid request.
- Pricing and model availability change frequently; verify them in provider documentation.
- Reasoning models may reject `temperature` or `max_tokens`; adapt the parameters if needed.
- This notebook makes real paid requests only when you run cells with configured keys.
